# Mammography Exercise

**Notebook 6 of 6, and optional.** Everything from notebooks 01 to 05, on a new problem:
**screening mammograms.** Each row is a small region of a mammogram, an X-ray image of a
breast, described by six numbers computed from the image. The target `calcification` is
1 where the region shows a microcalcification, a tiny calcium deposit that can be an early
sign of breast cancer and is followed up by a doctor.

About 2 regions in 100 are positive. This time a missed case is clearly the worse mistake,
and the problem turns out to be harder than the fraud data.

Each step has a cell marked `# your code`. Work top to bottom.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, fbeta_score,
    confusion_matrix, precision_recall_curve, roc_auc_score, average_precision_score,
)

In [2]:
df = pd.read_csv("data/mammography.csv")
df.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,calcification
0,0.230020,5.072578,-0.276061,0.832444,-0.377866,0.480322,0
1,0.155491,-0.169390,0.670652,-0.859553,-0.377866,-0.945723,0
2,-0.784415,-0.443654,5.674705,-0.859553,-0.377866,-0.945723,0
3,0.546088,0.131415,-0.456387,-0.859553,-0.377866,-0.945723,0
4,-0.102987,-0.394994,-0.140816,0.979703,-0.377866,1.013566,0


The six features come without names, only as image measurements, and they are already
standardized to mean 0 and standard deviation 1.

## Step 1: The imbalance and the split

How many regions are positive, as a count and as a share? Then separate features and
target, and split with `test_size=0.25`, `random_state=42` and stratification.

In [ ]:
# your code: the class balance, then X, y and the split
positiveRegions = df['calcification'].sum()

X = df.drop(columns='calcification')
y = df['calcification']

# split
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.25, random_state=42, stratify=y
)


## Step 2: The do-nothing baseline

What test accuracy does "never a calcification" reach, and what is its recall?

In [13]:
# your code: accuracy and recall of the baseline

# Create Nothing set
noPredection = np.zeros(len(X_test), dtype=int)

# DoNothingScores = [evaluate("nothing", noPredection)]
DoNothingScore = accuracy_score(y_test, noPredection)

np.round(DoNothingScore, 3) 

np.float64(0.977)

## Step 3: A plain logistic regression

Scale the features, train `LogisticRegression(max_iter=1000)`, and report its confusion
matrix, accuracy, precision, recall and F2 on the test set.

In [ ]:
# your code: scale, train, and report the five numbers

# scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

firstScore = accuracy_score(y_test, y_pred)
firstPrecision = precision_score(y_test, y_pred)
firstRecall = recall_score(y_test, y_pred)
firstf2 = fbeta_score(y_test, y_pred, beta=2)

print (f'accuracy = {firstScore}\n'
       f'precision = {firstPrecision}\n'
       f'recal = {firstRecall}\n'
       f'f2 = {firstf2}'
	   )

accuracy = 0.9824749642346209
precision = 0.75
recal = 0.36923076923076925
f2 = 0.410958904109589


In [30]:
def evaluate(name, pred):
    """Score test predictions with the three metrics of this notebook.

    Parameters
    ----------
    name : str
        A label for the recipe.
    pred : array-like
        Predicted classes for the test set, 0 or 1.

    Returns
    -------
    dict
        The recipe's name, number of alarms, precision, recall and F2.
    """
    return {
        "recipe": name,
        "alarms": int(pred.sum()),
        "accuracy" : accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F2": fbeta_score(y_test, pred, beta=2),
    }

> **Exercise:**
>
> No code needed. The model is 98 percent accurate. Would you let it screen mammograms?
> Which of the numbers above decides, and why?

Your answer here:

<details><summary>
Click here to compare your answer...
</summary>

No. **Recall is about 0.37**: the model misses almost two calcifications in three, and its
accuracy is barely above the 97.7 percent of doing nothing. For screening, recall is the
number that decides, with F2 as the single score, since a missed case is far worse than a
region a doctor looks at and dismisses.
</details>

## Step 4: Class weights

Train the same model with `class_weight="balanced"` and report precision, recall and F2.

In [31]:
# your code: the model with class weights
from sklearn.utils.class_weight import compute_class_weight

modelWeighed = LogisticRegression(max_iter=1000, class_weight='balanced')
modelWeighed.fit(X_train_scaled, y_train)

y_pred_weighed = modelWeighed.predict(X_test_scaled)
weighedResult = [evaluate("weighed model", y_pred_weighed)]
pd.DataFrame(weighedResult).round(3)

,recipe,alarms,accuracy,precision,recall,F2
0,weighed model,314,0.905,0.178,0.862,0.488


## Step 5: A threshold for a target recall

The screening team asks for a recall of **at least 0.8**. Choose the highest threshold
that reaches it **on the training set**, using the plain model's probabilities and
`precision_recall_curve` as in notebook 04. Then apply it to the test set once.

In [40]:
# your code: choose the threshold on the training set, then test it once
prob_train = model.predict_proba(X_train_scaled)[:, 1]
probPrecision, probRecall, probThreshholds = precision_recall_curve(y_train, prob_train)
candidates = pd.DataFrame(
    {"threshold": probThreshholds, "precision": probPrecision[:-1], "recall": probRecall[:-1]}
)
matchThreshhold = candidates[candidates['recall'] >= 0.8]['threshold'].max()

# apply threshold
prob_test = model.predict_proba(X_test_scaled)[:, 1]
pred_threshold = (prob_test >= matchThreshhold).astype(int)

thresholdResult = [evaluate("plain model, target recall 0.8", pred_threshold)]
pd.DataFrame(thresholdResult).round(3)




,recipe,alarms,accuracy,precision,recall,F2
0,"plain model, target recall 0.8",206,0.944,0.277,0.877,0.612


<details><summary>
Click here to compare your answer...
</summary>

The threshold is very low, about **0.04**, and on the test set recall is about **0.88**,
above the target this time, with precision about 0.28. Class weights reached a similar
recall, 0.86, but at a precision of 0.18. The chosen threshold gives the better trade and
the higher F2.

A precision of 0.28 means nearly four regions sent to a doctor for every real
calcification. For screening that is usually acceptable; a missed one is not.
</details>

## Step 6: The two summaries

Compute the plain model's ROC AUC and average precision on the test set, and compare each
with what random guessing would score.

In [ ]:
# your code: ROC AUC and average precision

<details><summary>
Click here to compare your answer...
</summary>

A ROC AUC of about **0.96** suggests an excellent model. An average precision of about
**0.62** says what steps 3 to 5 showed the hard way: finding the rare positives without
many false alarms is difficult here. The gap between the two is wider than on the fraud
data, because this problem is harder and the imbalance stronger.
</details>

## Summary

| Recipe | Precision | Recall | F2 |
| :-- | --: | --: | --: |
| Never a calcification | 0 | 0 | 0 |
| Plain model, threshold 0.5 | 0.75 | 0.37 | 0.41 |
| Class weights | 0.18 | 0.86 | 0.49 |
| Threshold chosen for recall 0.8 | 0.28 | 0.88 | 0.61 |

The same steps as on the fraud data, and a different problem. Accuracy said 98 percent;
recall said the model missed most cases; a metric chosen from the cost of each mistake,
and a threshold chosen on the training set, turned it into something a screening team
could use.